Data Loading into SQL Database

In this step, we load the cleaned dataset into a local SQLite database to enable SQL-based analysis. This allows us to combine Python preprocessing with SQL querying for deeper insights and better data handling.

We first import the necessary libraries, load the cleaned CSV file into a pandas DataFrame, and then create a connection to a SQLite database. Finally, we write the DataFrame into a SQL table called telco, replacing it if it already exists.

This step is important because it transforms the dataset into a structured relational format, making it easier to perform aggregations, grouping, and business-level analysis using SQL.

In [3]:
import sqlite3
import pandas as pd

# Load cleaned dataset
df = pd.read_csv("telco_churn_cleaned.csv")

# Create database connection
conn = sqlite3.connect("telco_churn.db")

# Load dataframe into SQL table
df.to_sql("telco", conn, if_exists="replace", index=False)

7032

Verifying Data Load in SQL

After loading the dataset into the SQLite database, we perform a quick verification step to ensure that the data has been correctly written into the telco table.

In this step, we run a simple SQL query to retrieve the first five rows of the table. This allows us to confirm that:

The table was created successfully
The data structure is intact
The values were loaded correctly without corruption or shifting

This is an important validation step before performing any further SQL analysis, as it ensures the database is correctly set up and ready for querying.

In [ ]:

query = "SELECT * FROM telco LIMIT 5;"
pd.read_sql(query, conn)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,Male,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,Male,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,Male,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


Overall Customer Churn Rate Analysis

In this step, we calculate the overall churn rate of the telecom customers using SQL aggregation functions.

We compute three key business metrics:

Total customers using COUNT(*)
Total churned customers using SUM(Churn) since churn is encoded as 1 (churned) and 0 (retained)
Churn rate (%) calculated as the proportion of churned customers out of the total customer base

The result is rounded to two decimal places for clarity and readability.

This step provides a high-level business KPI that helps us understand the overall customer retention performance of the company before breaking it down into deeper segments like contract type, payment method, and service usage.

In [ ]:
query = """
SELECT 
    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,
    ROUND(SUM(Churn) * 100.0 / COUNT(*), 2) AS churn_rate
FROM telco;
"""

pd.read_sql(query, conn)

,total_customers,churned_customers,churn_rate
0,7032,1869,26.58


Churn Analysis by Contract Type
In this step, we analyze how customer churn varies across different contract types using SQL grouping and aggregation.
We group the data by Contract type and calculate the following metrics for each category:


Total customers under each contract type


Number of churned customers using SUM(Churn) (where churn is encoded as 1)


Churn rate (%), calculated as the percentage of customers who left within each contract group


The results are sorted in descending order of churn rate to clearly highlight which contract types are associated with the highest customer loss.
This analysis helps identify the relationship between contract duration and customer retention, providing key business insight into which contract structures are most effective at reducing churn.

In [9]:
###CHURN BY CONTRACT

query = """
SELECT 
    Contract,
    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,
    ROUND(SUM(Churn) * 100.0 / COUNT(*), 2) AS churn_rate
FROM telco
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

pd.read_sql(query, conn)

,Contract,total_customers,churned_customers,churn_rate
0,Month-to-month,3875,1655,42.71
1,One year,1472,166,11.28
2,Two year,1685,48,2.85


Churn Analysis by Payment Method

In this step, we examine how customer churn varies based on the payment method used by customers.

We use SQL aggregation to group the dataset by PaymentMethod and compute key metrics for each category:

Total customers using each payment method
Number of churned customers using SUM(Churn) (where churn is encoded as 1)
Churn rate (%), calculated as the proportion of churned customers within each payment method group

The results are sorted in descending order of churn rate to clearly highlight which payment methods are associated with higher customer attrition.

This analysis helps identify whether certain payment methods are linked to customer dissatisfaction or inconvenience, providing valuable insight for improving billing systems and reducing churn.

In [ ]:

query = """
SELECT 
    PaymentMethod,
    COUNT(*) AS total_customers,
    SUM(Churn) AS churned_customers,
    ROUND(SUM(Churn) * 100.0 / COUNT(*), 2) AS churn_rate
FROM telco
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

pd.read_sql(query, conn)


,PaymentMethod,total_customers,churned_customers,churn_rate
0,Electronic check,2365,1071,45.29
1,Mailed check,1604,308,19.20
2,Bank transfer (automatic),1542,258,16.73
3,Credit card (automatic),1521,232,15.25
